# NHANES Full Dataset — Extensive EDA

**Goals:**
1. Which features are most relevant for `DPQ040` (fatigue severity)?
2. Which features best inform a user questionnaire to predict probability of any of the 15 diseases?
3. Which lab/blood markers are most relevant for each of the 15 diseases?

**Dataset:** `nhanes_merged_adults_final.csv` — 7,437 adults, 868 columns  
**Disease labels:** anemia, diabetes, thyroid, sleep_disorder, kidney, hepatitis_bc, liver, heart_failure, coronary_heart, emphysema_lungs, high_blood_pressure, high_cholesterol, menopause, overweight, alcohol

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import pointbiserialr, chi2_contingency, mannwhitneyu
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
PALETTE = 'Set2'

DATA_PATH = '../data/processed/nhanes_merged_adults_final.csv'
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

Shape: (7437, 868)


,SEQN,age_years,income_poverty_ratio,mec_exam_weight,interview_weight,survey_psu,survey_stratum,gender,ethnicity,education,...,LBXVTFT_blood_aaa_trifluorotoluene_ng_ml,LBDVFTLC_blood_aaa_trifluorotoluene_comment_code,LBXVTHF_blood_tetrahydrofuran_ng_ml,LBDVHTLC_blood_tetrahydrofuran_comment_code,LBXVTP_blood_1_2_3_trichloropropane_ng_ml,LBDVTPLC_blood_1_2_3_trichloropropane_comt_code,LBXVVB_blood_vinyl_bromide_ng_ml,LBDVVBLC_blood_vinyl_bromide_comment_code,LBXVXY_blood_m_p_xylene_ng_ml,LBDVXYLC_blood_m_p_xylene_comment_code
0,109266,29.0,5.00,8.154968e+03,7825.646112,2.0,168.0,Female,Non-Hispanic Asian,College graduate or above,...,0.028,1.0,0.088,1.0,0.028,1.0,0.032,1.0,0.035,0.0
1,109267,21.0,5.00,5.397605e-79,26379.991724,1.0,156.0,Female,Other Hispanic,Some college / AA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,109268,18.0,1.66,5.397605e-79,19639.221008,1.0,155.0,Female,Non-Hispanic White,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## 12. Point-Biserial Correlations: Non-Lab Features vs Each Disease

In [19]:
# Non-lab numeric features with decent coverage
non_lab_num = [c for c in df.select_dtypes(include='number').columns
               if c not in LAB_COLS + ['SEQN'] + DISEASE_COLS
               and df[c].notna().sum() > len(df) * 0.5]
print(f'Non-lab numeric features: {len(non_lab_num)}')

disease_feat_corrs = {}
for disease in DISEASE_COLS:
    corrs = []
    for feat in non_lab_num:
        sub = df[[feat, disease]].dropna()
        if len(sub) < 100 or sub[disease].std() == 0:
            continue
        r, p = pointbiserialr(sub[disease], sub[feat])
        corrs.append({'feature': feat, 'r': r, 'p': p})
    disease_feat_corrs[disease] = pd.DataFrame(corrs).sort_values('r', key=abs, ascending=False)

print('Top 5 non-lab correlates per disease:')
for disease, cdf in disease_feat_corrs.items():
    top5 = cdf.head(5)[['feature','r','p']]
    print(f'\n{disease.upper()}:')
    print(top5.to_string(index=False))

Non-lab numeric features: 150


Top 5 non-lab correlates per disease:

ANEMIA:
                                         feature         r            p
 mcq053___taking_treatment_for_anemia/past_3_mos -0.626782 0.000000e+00
                                       med_count  0.152421 6.762616e-40
huq071___overnight_hospital_patient_in_last_year -0.130539 1.254881e-29
                                       nan_count -0.130297 1.595882e-29
                              serum_albumin_g_dl -0.129422 2.829780e-25

DIABETES:
                                   feature         r             p
   diq160___ever_told_you_have_prediabetes -0.733488  0.000000e+00
                                 nan_count -0.440576  0.000000e+00
    diq010___doctor_told_you_have_diabetes -0.397776 1.911424e-280
                                 med_count  0.392860 5.296313e-273
mcq366b___doctor_told_to_increase_exercise -0.329836 3.126280e-188

THYROID:
                                         feature         r            p
     mcq160m___ever_told_y

---
## Feature Definition

In [ ]:
TARGET_DISEASES = [
    "high_blood_pressure",
    "alcohol",
    "hepatitis_bc",
    "heart_failure"
]